# 11 — Cross-Sectional XGBoost Ranker Tuning

This notebook searches for a stable combination of **feature columns** and
**XGBRanker hyperparameters** before the discretionary replay study.

The workflow deliberately stays exploratory:

- one broad `FeatureSetSpec` is calculated once;
- every trial slices columns from that loaded frame;
- Optuna tunes fine-grained ranker parameters on expanding folds inside outer train;
- top-performing trials provide fold-aggregated feature importance;
- each feature iteration removes a small low-importance fraction and randomly
  reintroduces individual excluded features;
- train-versus-fold metrics expose ordinary model overfitting;
- one explicitly selected configuration is fitted on complete outer train and
  checked once on outer validation;
- the locked test is never read.

The reported returns are research-target returns, not executable strategy P&L.
Keep the committed notebook generic and export material runs to HTML before
restoring the base state.


In [ ]:
from datetime import date
from pathlib import Path
from random import Random
from time import perf_counter

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import yaml
from optuna.samplers import TPESampler
from xgboost import XGBRanker

from swingtrader.core.paths import find_repo_root
from swingtrader.data import features
from swingtrader.data.bronze.queries import load_available_tickers
from swingtrader.data.db import resolve_database_engine
from swingtrader.modeling.datasets import (
    CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    CROSS_SECTIONAL_RETURN_TARGET_SET,
    TemporalDatasetSpec,
    UniverseSpec,
    build_temporal_dataset,
    to_tabular_dataset,
)
from swingtrader.modeling.experiments import (
    FixedTemporalSplitter,
    TemporalCrossValidationSpec,
    TemporalSplitSpec,
    build_expanding_temporal_folds,
)
from swingtrader.modeling.training import (
    deterministic_random_scores,
    evaluate_cross_sectional_scores,
    prepare_xgboost_ranking_data,
)

RANDOM_SEED = 23
TOP_K = 10
HORIZON = 5
TOP_QUANTILE_THRESHOLD = 0.80

N_FEATURE_ITERATIONS = 4
N_OPTUNA_TRIALS_PER_ITERATION = 6
N_IMPORTANCE_TRIALS = 3
FEATURE_DROP_FRACTION = 0.20
MIN_FEATURE_COUNT = 12
N_RANDOM_FEATURES_TO_ADD = 5

INCLUDE_SLOW_MARKET_STRUCTURE = False
PROVIDER = "yfinance"

DATA_START = date(2008, 1, 1)
TRAIN_START = date(2010, 1, 1)
TRAIN_END = date(2020, 12, 31)
VALIDATION_START = date(2021, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

CV_SPEC = TemporalCrossValidationSpec(
    n_folds=4,
    validation_sessions=252,
    minimum_train_sessions=1_260,
)

repo_root = find_repo_root()
database_url = (
    f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
)
ENGINE = resolve_database_engine(database_url=database_url)


## Resolve the exploratory universe

The notebook applies the current configured Stockholm Large/Mid Cap universe retrospectively. Results are therefore conditional on today's configured universe and may contain survivorship bias; they are not a reconstruction of historical point-in-time membership.


In [ ]:
def configured_tickers(path: Path) -> tuple[str, ...]:
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    return tuple(item["ticker"] for item in config["symbols"])


universe_directory = repo_root / "src" / "swingtrader" / "configs" / "universes"
configured = tuple(
    dict.fromkeys(
        configured_tickers(universe_directory / "se_large_cap.yml")
        + configured_tickers(universe_directory / "se_mid_cap.yml")
    )
)
available = set(
    load_available_tickers(
        engine=ENGINE,
        provider=PROVIDER,
        start_date=DATA_START,
        end_date=TEST_END,
    )
)
TICKERS = tuple(ticker for ticker in configured if ticker in available)

len(TICKERS), TICKERS[:5]


## Declare one broad feature set

Every enabled family is calculated once. Later trials only select columns from the resulting frame. The slow path-dependent market-structure block is disabled in the generic base notebook; enable it when its one-time loading cost is acceptable.


In [ ]:
feature_blocks = (
    features.FeatureBlockSpec(
        name="returns",
        builder=features.add_return_features,
        parameters={"horizons": (1, 5, 10, 20)},
        output_columns=("return_1d", "return_5d", "return_10d", "return_20d"),
        required_columns=frozenset({"adjusted_close"}),
    ),
    features.FeatureBlockSpec(
        name="cross_sectional",
        builder=features.add_cross_sectional_features,
        parameters={
            "return_horizons": (1, 5, 10, 20),
            "market_return_horizon": 1,
            "minimum_cross_section_size": 2,
        },
        output_columns=(
            "return_1d_cross_sectional_percentile",
            "return_5d_cross_sectional_percentile",
            "return_10d_cross_sectional_percentile",
            "return_20d_cross_sectional_percentile",
            "market_breadth_positive_1d",
            "market_mean_return_1d",
            "market_median_return_1d",
        ),
        required_columns=frozenset({"adjusted_close"}),
    ),
    features.FeatureBlockSpec(
        name="trend",
        builder=features.add_trend_features,
        parameters={
            "ma_lengths": (10, 20, 50),
            "adx_length": 14,
            "rolling_fraction_lookback": 20,
            "vwap_length": 20,
            "vwap_bollinger_length": 20,
            "vwap_bollinger_num_std": 2.0,
        },
        output_columns=(
            "ema_fast_to_ema_mid",
            "ema_mid_to_ema_slow",
            "ema_mid_to_sma_mid",
            "close_to_ema_fast",
            "close_to_ema_mid",
            "close_to_ema_slow",
            "close_over_ema_fast_fraction",
            "close_over_ema_mid_fraction",
            "close_over_ema_slow_fraction",
            "adx",
            "plus_di",
            "minus_di",
            "vwap_distance",
            "vwap_distance_percent_b",
        ),
        required_columns=frozenset(
            {"high", "low", "close", "volume", "adjusted_close"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="momentum",
        builder=features.add_momentum_features,
        parameters={
            "ppo_lengths": (12, 26, 9),
            "ppo_percentile_min_history": 100,
            "rsi_length": 21,
            "rsi_bollinger_length": 20,
            "rsi_bollinger_num_std": 2.0,
            "stochastic_k_length": 14,
            "stochastic_k_smoothing": 3,
            "stochastic_d_length": 3,
            "mfi_length": 14,
            "mfi_bollinger_length": 20,
            "mfi_bollinger_num_std": 2.0,
            "squeeze_bb_length": 20,
            "squeeze_bb_mult": 2.0,
            "squeeze_kc_length": 20,
            "squeeze_kc_mult": 1.5,
            "squeeze_atr_length": 14,
        },
        output_columns=(
            "ppo",
            "ppo_signal",
            "ppo_histogram",
            "ppo_percentile",
            "rsi",
            "rsi_percent_b",
            "stochastic_k",
            "stochastic_d",
            "mfi",
            "mfi_percent_b",
            "squeeze_on",
            "squeeze_off",
            "squeeze_released",
            "squeeze_width_ratio",
            "squeeze_momentum_atr",
            "squeeze_momentum_atr_change",
            "squeeze_duration",
            "squeeze_release_duration",
        ),
        required_columns=frozenset(
            {"high", "low", "close", "adjusted_close", "volume"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="volatility",
        builder=features.add_volatility_features,
        parameters={
            "adr_length": 20,
            "atr_length": 14,
            "bollinger_length": 20,
            "bollinger_num_std": 2.0,
        },
        output_columns=(
            "adr_percent",
            "atr_percent",
            "bollinger_bandwidth",
            "bollinger_percent_b",
        ),
        required_columns=frozenset({"high", "low", "close", "adjusted_close"}),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="price_action",
        builder=features.add_price_action_features,
        parameters={
            "atr_length": 14,
            "range_percentile_length": 20,
            "breakout_length": 20,
            "rolling_candle_lookback": 14,
        },
        output_columns=(
            "candle_signed_body_fraction",
            "candle_upper_wick_fraction",
            "candle_lower_wick_fraction",
            "candle_close_location",
            "candle_range_atr",
            "candle_gap_atr",
            "range_percentile_20",
            "candle_inside_bar",
            "candle_outside_bar",
            "candle_engulfing_strength",
            "candle_lower_rejection_strength",
            "candle_upper_rejection_strength",
            "candle_consecutive_inside_bars",
            "candle_direction_run",
            "candle_direction_run_return",
            "candle_direction_run_body_atr",
            "candle_close_to_prior_high_atr_20",
            "candle_close_to_prior_low_atr_20",
            "candle_breakout_high_strength_20",
            "candle_breakout_low_strength_20",
            "candle_failed_breakout_high_strength_20",
            "candle_failed_breakout_low_strength_20",
            "rolling_bullish_candle_fraction",
        ),
        required_columns=frozenset(
            {"open", "high", "low", "close", "adjusted_close"}
        ),
        history_requirement=features.HistoryRequirement.EXPANDING,
    ),
    features.FeatureBlockSpec(
        name="volume",
        builder=features.add_volume_features,
        parameters={"turnover_zscore_length": 252, "turnover_zscore_log": True},
        output_columns=("turnover_zscore",),
        required_columns=frozenset({"close", "volume"}),
    ),
)

if INCLUDE_SLOW_MARKET_STRUCTURE:
    feature_blocks += (
        features.FeatureBlockSpec(
            name="market_structure",
            builder=features.add_market_structure_features,
            parameters={
                "donchian_length": 20,
                "zigzag_deviation": 5.0,
                "zigzag_pivot_legs": 10,
                "zigzag_consistency_pivots": 4,
                "zigzag_dynamics_legs": 6,
                "zigzag_atr_length": 14,
            },
            output_columns=(
                "donchian_position",
                "zigzag_last_direction",
                "zigzag_last_swing_return",
                "zigzag_last_swing_bars",
                "zigzag_swing_return_per_bar",
                "zigzag_bars_since_pivot",
                "zigzag_retracement",
                "market_structure_high_change",
                "market_structure_low_change",
                "market_structure_high_rate",
                "market_structure_low_rate",
                "market_structure_high_consistency",
                "market_structure_low_consistency",
                "market_structure_leg_balance",
                "market_structure_efficiency",
                "market_structure_close_to_prior_high_atr",
                "market_structure_close_to_prior_low_atr",
                "market_structure_breakout_high_strength",
                "market_structure_breakout_low_strength",
                "market_structure_failed_breakout_high_strength",
                "market_structure_failed_breakout_low_strength",
            ),
            required_columns=frozenset({"high", "low", "close"}),
            history_requirement=features.HistoryRequirement.PATH_DEPENDENT,
        ),
    )

BROAD_FEATURE_SET = features.FeatureSetSpec(
    name="cross_sectional_ranker_tuning_candidates",
    version="1",
    blocks=feature_blocks,
)

len(BROAD_FEATURE_SET.feature_columns), BROAD_FEATURE_SET.feature_columns[:8]


## Build and split the dataset once

`DATA_START` begins before `TRAIN_START` to provide warm-up history without loading the complete database. Expanding and path-dependent features are conditional on this chosen start date, so keep it fixed when comparing runs. Dataset construction is the expensive step; all tuning trials reuse the resulting frames.


In [ ]:
universe = UniverseSpec(
    name="stockholm_large_mid_cap_ranker_tuning",
    version="1",
    provider=PROVIDER,
    tickers=TICKERS,
)
dataset_spec = TemporalDatasetSpec(
    feature_set=BROAD_FEATURE_SET,
    target_set=CROSS_SECTIONAL_RETURN_TARGET_SET,
    task=CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=DATA_START,
    data_end=TEST_END,
)
split_spec = TemporalSplitSpec(
    name="cross_sectional_ranker_tuning_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

started = perf_counter()
bundle = build_temporal_dataset(engine=ENGINE, spec=dataset_spec)
split_result = FixedTemporalSplitter(split_spec).assign(bundle)
load_seconds = perf_counter() - started

tabular = to_tabular_dataset(bundle)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

{
    "load_seconds": round(load_seconds, 1),
    "features": bundle.features.shape[1],
    "bundle_rows": len(bundle.features),
    "train_rows": len(train_positions),
    "validation_rows": len(validation_positions),
}


In [ ]:
relative_return_column = f"market_relative_forward_return_{HORIZON}d"
percentile_column = f"forward_return_{HORIZON}d_cross_sectional_percentile"
relevance_column = f"forward_return_{HORIZON}d_relevance_grade"

relative_return = bundle.targets[relative_return_column]
percentile = bundle.targets[percentile_column]
relevance = bundle.targets[relevance_column]

folds = build_expanding_temporal_folds(bundle, split_result, spec=CV_SPEC)
pd.DataFrame(
    [
        {
            "fold": fold.number,
            "train_start": fold.train_start,
            "train_end": fold.train_end,
            "validation_start": fold.validation_start,
            "validation_end": fold.validation_end,
            "train_rows": len(fold.train_indices),
            "validation_rows": len(fold.validation_indices),
        }
        for fold in folds
    ]
)


## Define the initial feature selection

The search starts from one interpretable column set rather than coarse feature-family
ablation. Later iterations prune individual columns and sample replacements from every
feature already present in the broad loaded frame.

`PROTECTED_FEATURE_COLUMNS` is empty by default. Add only features that should never be
removed for a specific study.


In [ ]:
loaded_columns = tuple(tabular.X.columns)

INITIAL_FEATURE_COLUMNS = (
    "return_10d",
    "return_20d",
    "return_5d_cross_sectional_percentile",
    "return_10d_cross_sectional_percentile",
    "return_20d_cross_sectional_percentile",
    "market_breadth_positive_1d",
    "ema_fast_to_ema_mid",
    "ema_mid_to_ema_slow",
    "close_to_ema_fast",
    "close_to_ema_mid",
    "close_to_ema_slow",
    "ppo",
    "rsi",
    "rsi_percent_b",
    "atr_percent",
    "bollinger_percent_b",
    "candle_range_atr",
    "candle_close_to_prior_high_atr_20",
    "candle_breakout_high_strength_20",
    "rolling_bullish_candle_fraction",
)
PROTECTED_FEATURE_COLUMNS: tuple[str, ...] = ()

missing_initial = sorted(set(INITIAL_FEATURE_COLUMNS) - set(loaded_columns))
missing_protected = sorted(set(PROTECTED_FEATURE_COLUMNS) - set(loaded_columns))
if missing_initial:
    raise ValueError(f"Initial feature columns are not loaded: {missing_initial}")
if missing_protected:
    raise ValueError(f"Protected feature columns are not loaded: {missing_protected}")
if not set(PROTECTED_FEATURE_COLUMNS).issubset(INITIAL_FEATURE_COLUMNS):
    raise ValueError("Protected feature columns must be included initially.")
if len(INITIAL_FEATURE_COLUMNS) < MIN_FEATURE_COUNT:
    raise ValueError("Initial feature selection is smaller than MIN_FEATURE_COUNT.")

{
    "loaded_features": len(loaded_columns),
    "initial_features": len(INITIAL_FEATURE_COLUMNS),
    "protected_features": len(PROTECTED_FEATURE_COLUMNS),
}


## Define the Optuna search

Each feature iteration runs a small multi-objective Optuna study. The two objectives are:

1. mean market-relative forward return among the daily top `k`;
2. mean future cross-sectional percentile among the daily top `k`.

The parameter ranges are intentionally fine enough to test nearby learning rates,
estimator counts, and tree depths. Increase the visible trial and iteration counts when
runtime permits.

Feature importance is averaged across folds for each trial, then across the best few
trials in the iteration. The next feature selection drops only a small bottom fraction
before sampling individual replacements. Gain importance is treated as a search guide,
not a definitive statement that low-importance features have no value.


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

FIXED_RANKER_PARAMETERS = {
    "objective": "rank:ndcg",
    "tree_method": "hist",
    "lambdarank_pair_method": "topk",
    "lambdarank_num_pair_per_sample": TOP_K,
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
    "verbosity": 0,
}


def suggest_ranker_parameters(trial: optuna.Trial) -> dict[str, object]:
    """Suggest one fine-grained XGBRanker parameter configuration."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 450, step=20),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.02,
            0.12,
            step=0.01,
        ),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0, step=0.05),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.50,
            1.0,
            step=0.05,
        ),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0, step=0.05),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0, step=0.10),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 15.0, step=1.0),
        "lambdarank_normalization": trial.suggest_categorical(
            "lambdarank_normalization",
            [True, False],
        ),
    }


## Top-tail evaluation

The tuning metrics emphasize the selected shortlist rather than the complete universe:

- mean future cross-sectional percentile among the daily top `k`;
- mean and median market-relative forward return among the daily top `k`;
- fraction of selected stocks that finish in the future top quintile;
- fraction of dates where the selected basket has positive market-relative return.

The same metrics are calculated on each fold's training and validation rows. Their
difference is a direct generalization-gap diagnostic. The numeric XGBRanker score itself
is not calibrated and is not used as a fixed threshold.


In [ ]:
def evaluate_top_tail(
    scores: pd.Series,
    future_percentile: pd.Series,
    future_relative_return: pd.Series,
    *,
    top_k: int,
    random_seed: int,
) -> tuple[pd.Series, pd.DataFrame]:
    """Summarize daily top-k percentile and market-relative outcomes."""
    if not scores.index.equals(future_percentile.index):
        raise ValueError("Scores and future percentiles must share an index.")
    if not scores.index.equals(future_relative_return.index):
        raise ValueError("Scores and future returns must share an index.")

    frame = pd.DataFrame(
        {
            "score": scores.astype("float64"),
            "future_percentile": future_percentile.astype("float64"),
            "future_relative_return": future_relative_return.astype("float64"),
        }
    )
    frame["tiebreak"] = deterministic_random_scores(
        frame.index,
        seed=random_seed,
    )

    rows = []
    for (provider, trading_date), group in frame.groupby(
        level=["provider", "trading_date"],
        sort=False,
    ):
        selected = group.sort_values(
            ["score", "tiebreak"],
            ascending=False,
            kind="stable",
        ).head(min(top_k, len(group)))
        rows.append(
            {
                "provider": provider,
                "trading_date": trading_date,
                "selected_count": len(selected),
                "top_k_mean_percentile": selected["future_percentile"].mean(),
                "top_k_mean_relative_return": selected[
                    "future_relative_return"
                ].mean(),
                "top_k_top_quintile_fraction": selected[
                    "future_percentile"
                ].ge(TOP_QUANTILE_THRESHOLD).mean(),
            }
        )

    daily = pd.DataFrame(rows).set_index(["provider", "trading_date"]).sort_index()
    summary = pd.Series(
        {
            "date_count": float(len(daily)),
            "mean_top_k_percentile": daily["top_k_mean_percentile"].mean(),
            "mean_top_k_relative_return": daily[
                "top_k_mean_relative_return"
            ].mean(),
            "median_top_k_relative_return": daily[
                "top_k_mean_relative_return"
            ].median(),
            "mean_top_k_top_quintile_fraction": daily[
                "top_k_top_quintile_fraction"
            ].mean(),
            "positive_top_k_date_fraction": daily[
                "top_k_mean_relative_return"
            ].gt(0).mean(),
        },
        dtype="float64",
    )
    return summary, daily


## Run the iterative feature and hyperparameter search

Progress is printed after every Optuna trial and every completed feature iteration.
With the default settings, the study runs:

`N_FEATURE_ITERATIONS × N_OPTUNA_TRIALS_PER_ITERATION × CV_SPEC.n_folds`

ranker fits, plus predictions on the corresponding fold-training rows.


In [ ]:
def evaluate_ranker_trial(
    trial: optuna.Trial,
    feature_columns: tuple[str, ...],
    *,
    feature_iteration: int,
) -> tuple[float, float]:
    """Evaluate one Optuna trial across the configured expanding folds."""
    model_parameters = suggest_ranker_parameters(trial)
    fold_metrics = []
    fold_importances = []

    for fold in folds:
        X_fold_train = tabular.X.iloc[fold.train_indices].loc[
            :,
            list(feature_columns),
        ]
        X_fold_validation = tabular.X.iloc[fold.validation_indices].loc[
            :,
            list(feature_columns),
        ]
        relevance_fold_train = relevance.iloc[fold.train_indices]
        relevance_fold_validation = relevance.iloc[fold.validation_indices]

        X_rank_train, y_rank_train, qid_train = prepare_xgboost_ranking_data(
            X_fold_train,
            relevance_fold_train,
        )
        X_rank_validation, y_rank_validation, _ = prepare_xgboost_ranking_data(
            X_fold_validation,
            relevance_fold_validation,
        )

        model = XGBRanker(
            **FIXED_RANKER_PARAMETERS,
            **model_parameters,
        )
        model.fit(
            X_rank_train,
            y_rank_train,
            qid=qid_train,
            verbose=False,
        )

        train_scores = pd.Series(
            model.predict(X_rank_train),
            index=X_rank_train.index,
            dtype="float64",
            name="score",
        )
        validation_scores = pd.Series(
            model.predict(X_rank_validation),
            index=X_rank_validation.index,
            dtype="float64",
            name="score",
        )

        train_summary, _ = evaluate_top_tail(
            train_scores,
            percentile.iloc[fold.train_indices].reindex(train_scores.index),
            relative_return.iloc[fold.train_indices].reindex(train_scores.index),
            top_k=TOP_K,
            random_seed=RANDOM_SEED,
        )
        validation_summary, _ = evaluate_top_tail(
            validation_scores,
            percentile.iloc[fold.validation_indices].reindex(
                validation_scores.index
            ),
            relative_return.iloc[fold.validation_indices].reindex(
                validation_scores.index
            ),
            top_k=TOP_K,
            random_seed=RANDOM_SEED,
        )

        fold_metrics.append(
            {
                "fold": int(fold.number),
                "train_top_k_percentile": float(
                    train_summary["mean_top_k_percentile"]
                ),
                "validation_top_k_percentile": float(
                    validation_summary["mean_top_k_percentile"]
                ),
                "train_top_k_relative_return": float(
                    train_summary["mean_top_k_relative_return"]
                ),
                "validation_top_k_relative_return": float(
                    validation_summary["mean_top_k_relative_return"]
                ),
                "validation_median_top_k_relative_return": float(
                    validation_summary["median_top_k_relative_return"]
                ),
                "validation_top_quintile_fraction": float(
                    validation_summary["mean_top_k_top_quintile_fraction"]
                ),
                "validation_positive_date_fraction": float(
                    validation_summary["positive_top_k_date_fraction"]
                ),
            }
        )
        fold_importances.append(
            pd.Series(
                model.feature_importances_,
                index=feature_columns,
                dtype="float64",
            )
        )

    fold_frame = pd.DataFrame(fold_metrics)
    mean_importance = pd.concat(fold_importances, axis=1).mean(axis=1)

    summary = {
        "mean_train_top_k_percentile": float(
            fold_frame["train_top_k_percentile"].mean()
        ),
        "mean_top_k_percentile": float(
            fold_frame["validation_top_k_percentile"].mean()
        ),
        "percentile_generalization_gap": float(
            (
                fold_frame["train_top_k_percentile"]
                - fold_frame["validation_top_k_percentile"]
            ).mean()
        ),
        "mean_train_top_k_relative_return": float(
            fold_frame["train_top_k_relative_return"].mean()
        ),
        "mean_top_k_relative_return": float(
            fold_frame["validation_top_k_relative_return"].mean()
        ),
        "relative_return_generalization_gap": float(
            (
                fold_frame["train_top_k_relative_return"]
                - fold_frame["validation_top_k_relative_return"]
            ).mean()
        ),
        "mean_fold_median_top_k_relative_return": float(
            fold_frame["validation_median_top_k_relative_return"].mean()
        ),
        "std_fold_top_k_relative_return": float(
            fold_frame["validation_top_k_relative_return"].std(ddof=0)
        ),
        "worst_fold_top_k_relative_return": float(
            fold_frame["validation_top_k_relative_return"].min()
        ),
        "mean_top_k_top_quintile_fraction": float(
            fold_frame["validation_top_quintile_fraction"].mean()
        ),
        "mean_positive_top_k_date_fraction": float(
            fold_frame["validation_positive_date_fraction"].mean()
        ),
    }

    trial.set_user_attr("feature_iteration", feature_iteration)
    trial.set_user_attr("feature_columns", list(feature_columns))
    trial.set_user_attr("fold_metrics", fold_metrics)
    trial.set_user_attr(
        "feature_importance",
        {column: float(value) for column, value in mean_importance.items()},
    )
    trial.set_user_attr("summary", summary)

    return (
        summary["mean_top_k_relative_return"],
        summary["mean_top_k_percentile"],
    )


def add_selection_rank(frame: pd.DataFrame) -> pd.DataFrame:
    """Add the notebook's scale-free ranking columns to a trial table."""
    ranked = frame.copy()
    ranked["return_rank"] = ranked["mean_top_k_relative_return"].rank(
        ascending=False,
        method="min",
    )
    ranked["percentile_rank"] = ranked["mean_top_k_percentile"].rank(
        ascending=False,
        method="min",
    )
    ranked["worst_fold_rank"] = ranked[
        "worst_fold_top_k_relative_return"
    ].rank(ascending=False, method="min")
    ranked["selection_rank"] = ranked[
        ["return_rank", "percentile_rank", "worst_fold_rank"]
    ].mean(axis=1)
    return ranked.sort_values(
        ["selection_rank", "mean_top_k_relative_return"],
        ascending=[True, False],
    ).reset_index(drop=True)


def select_next_feature_columns(
    current_columns: tuple[str, ...],
    importance: pd.Series,
    *,
    feature_iteration: int,
) -> tuple[tuple[str, ...], tuple[str, ...], tuple[str, ...]]:
    """Prune low-importance columns and sample deterministic replacements."""
    protected = set(PROTECTED_FEATURE_COLUMNS)
    droppable = [column for column in current_columns if column not in protected]
    maximum_drop = max(0, len(current_columns) - MIN_FEATURE_COUNT)
    requested_drop = max(1, round(len(current_columns) * FEATURE_DROP_FRACTION))
    drop_count = min(maximum_drop, requested_drop, len(droppable))

    dropped = tuple(
        importance.reindex(droppable)
        .fillna(0.0)
        .sort_values()
        .head(drop_count)
        .index
    )
    retained = set(current_columns) - set(dropped)

    replacement_pool = [
        column for column in loaded_columns if column not in retained
    ]
    add_count = min(
        N_RANDOM_FEATURES_TO_ADD,
        len(dropped),
        len(replacement_pool),
    )
    added = tuple(
        Random(RANDOM_SEED + feature_iteration).sample(
            replacement_pool,
            add_count,
        )
    )

    selected = retained | set(added)
    next_columns = tuple(
        column for column in loaded_columns if column in selected
    )
    return next_columns, dropped, added


def make_progress_callback(feature_iteration: int):
    """Create a callback that prints trial progress for one feature iteration."""

    def report_progress(
        study: optuna.Study,
        trial: optuna.trial.FrozenTrial,
    ) -> None:
        del study
        if trial.values is None:
            return
        summary = trial.user_attrs["summary"]
        print(
            f"Feature iteration ({feature_iteration} / {N_FEATURE_ITERATIONS}), "
            f"Optuna trial ({trial.number + 1} / "
            f"{N_OPTUNA_TRIALS_PER_ITERATION}): "
            f"top-{TOP_K} return="
            f"{summary['mean_top_k_relative_return']:.3%}, "
            f"percentile={summary['mean_top_k_percentile']:.3f}, "
            f"return gap="
            f"{summary['relative_return_generalization_gap']:.3%}"
        )

    return report_progress


In [ ]:
studies = []
trial_rows = []
fold_rows = []
iteration_rows = []
feature_history = []
trial_configs: dict[int, dict[str, object]] = {}

current_columns = INITIAL_FEATURE_COLUMNS
next_trial_id = 1

for feature_iteration in range(1, N_FEATURE_ITERATIONS + 1):
    print(
        f"Starting feature iteration "
        f"({feature_iteration} / {N_FEATURE_ITERATIONS}) "
        f"with {len(current_columns)} features."
    )

    study = optuna.create_study(
        directions=["maximize", "maximize"],
        sampler=TPESampler(
            seed=RANDOM_SEED + feature_iteration,
            n_startup_trials=2,
        ),
    )
    studies.append(study)
    study.optimize(
        lambda trial: evaluate_ranker_trial(
            trial,
            current_columns,
            feature_iteration=feature_iteration,
        ),
        n_trials=N_OPTUNA_TRIALS_PER_ITERATION,
        callbacks=[make_progress_callback(feature_iteration)],
        show_progress_bar=False,
    )

    iteration_trial_rows = []
    trial_number_to_id = {}
    for trial in study.trials:
        if trial.state is not optuna.trial.TrialState.COMPLETE:
            continue

        trial_id = next_trial_id
        next_trial_id += 1
        trial_number_to_id[trial.number] = trial_id

        summary = trial.user_attrs["summary"]
        row = {
            "trial_id": trial_id,
            "feature_iteration": feature_iteration,
            "optuna_trial_number": trial.number,
            "feature_count": len(current_columns),
            **summary,
            **trial.params,
        }
        iteration_trial_rows.append(row)
        trial_rows.append(row)
        trial_configs[trial_id] = {
            "feature_iteration": feature_iteration,
            "feature_columns": current_columns,
            "parameters": dict(trial.params),
        }

        for fold_row in trial.user_attrs["fold_metrics"]:
            fold_rows.append(
                {
                    "trial_id": trial_id,
                    "feature_iteration": feature_iteration,
                    **fold_row,
                }
            )

    iteration_frame = add_selection_rank(pd.DataFrame(iteration_trial_rows))
    top_trials = iteration_frame.head(
        min(N_IMPORTANCE_TRIALS, len(iteration_frame))
    )
    study_trials = {trial.number: trial for trial in study.trials}
    importance = pd.concat(
        [
            pd.Series(
                study_trials[int(row.optuna_trial_number)].user_attrs[
                    "feature_importance"
                ],
                dtype="float64",
            )
            for row in top_trials.itertuples()
        ],
        axis=1,
    ).mean(axis=1)
    importance = importance.reindex(current_columns).fillna(0.0).sort_values(
        ascending=False
    )

    best = iteration_frame.iloc[0]
    next_columns = current_columns
    dropped: tuple[str, ...] = ()
    added: tuple[str, ...] = ()
    if feature_iteration < N_FEATURE_ITERATIONS:
        next_columns, dropped, added = select_next_feature_columns(
            current_columns,
            importance,
            feature_iteration=feature_iteration,
        )

    iteration_rows.append(
        {
            "feature_iteration": feature_iteration,
            "feature_count": len(current_columns),
            "best_trial_id": int(best["trial_id"]),
            "best_mean_top_k_relative_return": best[
                "mean_top_k_relative_return"
            ],
            "best_mean_top_k_percentile": best["mean_top_k_percentile"],
            "best_relative_return_generalization_gap": best[
                "relative_return_generalization_gap"
            ],
            "next_feature_count": len(next_columns),
            "dropped_features": dropped,
            "added_features": added,
        }
    )
    feature_history.append(
        {
            "feature_iteration": feature_iteration,
            "feature_columns": current_columns,
            "importance": importance,
            "dropped_features": dropped,
            "added_features": added,
        }
    )

    print(
        f"Completed feature iteration "
        f"({feature_iteration} / {N_FEATURE_ITERATIONS}): "
        f"best top-{TOP_K} return="
        f"{best['mean_top_k_relative_return']:.3%}, "
        f"percentile={best['mean_top_k_percentile']:.3f}, "
        f"features={len(current_columns)} -> {len(next_columns)}."
    )
    if dropped or added:
        print(f"Dropped: {dropped}")
        print(f"Added: {added}")

    current_columns = next_columns

fold_results = pd.DataFrame(fold_rows)
iteration_summary = pd.DataFrame(iteration_rows)
leaderboard = add_selection_rank(pd.DataFrame(trial_rows))

iteration_summary


### Compare with a deterministic random shortlist

This baseline is calculated on the same inner-fold validation rows. It gives direct
context for the top-tail percentile and relative-return levels.


In [ ]:
random_rows = []
for fold in folds:
    fold_index = tabular.X.iloc[fold.validation_indices].index
    random_scores = deterministic_random_scores(fold_index, seed=RANDOM_SEED)
    summary, _ = evaluate_top_tail(
        random_scores,
        percentile.iloc[fold.validation_indices],
        relative_return.iloc[fold.validation_indices],
        top_k=TOP_K,
        random_seed=RANDOM_SEED,
    )
    random_rows.append({"fold": fold.number, **summary.to_dict()})

random_fold_results = pd.DataFrame(random_rows)
random_fold_results


## Inspect stability and overfitting before selecting a trial

The combined `selection_rank` gives equal weight to mean top-`k` relative return,
mean top-`k` percentile, and worst-fold relative return. It is only a scale-free EDA
convenience.

Prefer configurations that:

- perform sensibly in every fold;
- have a moderate train-versus-validation generalization gap;
- are not dependent on one exact feature iteration or hyperparameter combination.


In [ ]:
display_columns = [
    "trial_id",
    "feature_iteration",
    "feature_count",
    "selection_rank",
    "mean_top_k_relative_return",
    "mean_fold_median_top_k_relative_return",
    "mean_top_k_percentile",
    "worst_fold_top_k_relative_return",
    "std_fold_top_k_relative_return",
    "relative_return_generalization_gap",
    "percentile_generalization_gap",
    "mean_top_k_top_quintile_fraction",
    "n_estimators",
    "learning_rate",
    "max_depth",
]
leaderboard.loc[:, display_columns].head(20)


In [ ]:
top_trial_ids = leaderboard.head(8)["trial_id"]
plot_frame = fold_results[fold_results["trial_id"].isin(top_trial_ids)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for trial_id, group in plot_frame.groupby("trial_id"):
    iteration = group["feature_iteration"].iloc[0]
    label = f"{trial_id} (iteration {iteration})"
    axes[0].plot(
        group["fold"],
        group["validation_top_k_relative_return"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        group["fold"],
        group["validation_top_k_percentile"],
        marker="o",
        label=label,
    )

axes[0].axhline(
    random_fold_results["mean_top_k_relative_return"].mean(),
    linestyle="--",
    label="random mean",
)
axes[1].axhline(
    random_fold_results["mean_top_k_percentile"].mean(),
    linestyle="--",
    label="random mean",
)
axes[0].set_title(f"Top-{TOP_K} market-relative return by fold")
axes[1].set_title(f"Top-{TOP_K} future percentile by fold")
for axis in axes:
    axis.set_xlabel("Fold")
    axis.legend(fontsize=8)
plt.tight_layout()


## Choose one trial and evaluate outer validation once

Inspect the inner-fold leaderboard, feature history, and generalization gaps first.
Then set `CHOSEN_TRIAL_ID` explicitly. Leaving it as `None` stops a normal
**Run All** before outer validation is consumed.

Do not repeatedly change the search after reading outer-validation results.


In [ ]:
CHOSEN_TRIAL_ID: int | None = 24

if CHOSEN_TRIAL_ID is None:
    raise ValueError(
        "Set CHOSEN_TRIAL_ID explicitly after selecting from inner-fold results."
    )
if CHOSEN_TRIAL_ID not in trial_configs:
    raise ValueError(f"Unknown CHOSEN_TRIAL_ID: {CHOSEN_TRIAL_ID}")

chosen_row = leaderboard.set_index("trial_id").loc[CHOSEN_TRIAL_ID]
chosen_config = trial_configs[CHOSEN_TRIAL_ID]
chosen_columns = tuple(chosen_config["feature_columns"])
chosen_parameters = dict(chosen_config["parameters"])

{
    "trial": chosen_row.to_dict(),
    "feature_columns": chosen_columns,
    "parameters": chosen_parameters,
}


In [ ]:
X_outer_train = tabular.X.iloc[train_positions].loc[:, list(chosen_columns)]
X_outer_validation = tabular.X.iloc[validation_positions].loc[
    :, list(chosen_columns)
]

X_rank_train, y_rank_train, qid_train = prepare_xgboost_ranking_data(
    X_outer_train,
    relevance.iloc[train_positions],
)
X_rank_validation, y_rank_validation, _ = prepare_xgboost_ranking_data(
    X_outer_validation,
    relevance.iloc[validation_positions],
)

chosen_model = XGBRanker(
    **FIXED_RANKER_PARAMETERS,
    **chosen_parameters,
)
chosen_model.fit(
    X_rank_train,
    y_rank_train,
    qid=qid_train,
    verbose=False,
)
validation_scores = pd.Series(
    chosen_model.predict(X_rank_validation),
    index=X_rank_validation.index,
    dtype="float64",
    name="score",
)

ranking_summary, ranking_daily = evaluate_cross_sectional_scores(
    validation_scores,
    y_rank_validation,
    relative_return.iloc[validation_positions].reindex(validation_scores.index),
    top_k=TOP_K,
    random_seed=RANDOM_SEED,
)
top_tail_summary, top_tail_daily = evaluate_top_tail(
    validation_scores,
    percentile.iloc[validation_positions].reindex(validation_scores.index),
    relative_return.iloc[validation_positions].reindex(validation_scores.index),
    top_k=TOP_K,
    random_seed=RANDOM_SEED,
)

pd.concat(
    {
        "whole_ranking": ranking_summary,
        "top_tail": top_tail_summary,
    },
    axis=1,
)


### Outer-validation stability by year


In [ ]:
validation_by_year = top_tail_daily.assign(
    year=top_tail_daily.index.get_level_values("trading_date").year
).groupby("year").agg(
    date_count=("top_k_mean_relative_return", "size"),
    mean_top_k_percentile=("top_k_mean_percentile", "mean"),
    mean_top_k_relative_return=("top_k_mean_relative_return", "mean"),
    median_top_k_relative_return=("top_k_mean_relative_return", "median"),
    positive_top_k_date_fraction=(
        "top_k_mean_relative_return",
        lambda values: values.gt(0).mean(),
    ),
    mean_top_k_top_quintile_fraction=(
        "top_k_top_quintile_fraction",
        "mean",
    ),
)
validation_by_year


### Inspect the extreme score tail

Ranker scores are not probabilities and their absolute scale can move when features or parameters change. The table therefore converts scores to a within-date percentile before comparing outcomes.


In [ ]:
validation_outcomes = pd.DataFrame(
    {
        "score": validation_scores,
        "future_percentile": percentile.iloc[validation_positions].reindex(
            validation_scores.index
        ),
        "future_relative_return": relative_return.iloc[validation_positions].reindex(
            validation_scores.index
        ),
    }
)
validation_outcomes["daily_score_percentile"] = validation_outcomes.groupby(
    level=["provider", "trading_date"]
)["score"].rank(method="average", pct=True)
validation_outcomes["score_bucket"] = pd.cut(
    validation_outcomes["daily_score_percentile"],
    bins=[0.0, 0.50, 0.80, 0.90, 0.95, 0.98, 1.0],
    include_lowest=True,
)

score_tail_summary = validation_outcomes.groupby(
    "score_bucket",
    observed=True,
).agg(
    observations=("score", "size"),
    mean_future_percentile=("future_percentile", "mean"),
    mean_future_relative_return=("future_relative_return", "mean"),
    median_future_relative_return=("future_relative_return", "median"),
    positive_relative_return_fraction=(
        "future_relative_return",
        lambda values: values.gt(0).mean(),
    ),
)
score_tail_summary


### Inspect the selected model's feature importance


In [ ]:
importance = pd.Series(
    chosen_model.feature_importances_,
    index=chosen_columns,
    name="importance",
).sort_values(ascending=False)

importance.head(25).sort_values().plot.barh(figsize=(9, 8))
plt.title("Chosen XGBRanker feature importance")
plt.xlabel("Gain importance")
plt.tight_layout()

importance.head(25)


# Check prediction in timeseries

In [ ]:
tickers = validation_scores.index.get_level_values("ticker").unique()
start_date = validation_scores.index.get_level_values("trading_date").min()
end_date = validation_scores.index.get_level_values("trading_date").max()
tickers, start_date, end_date

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from swingtrader import indicators
from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.core.numerical import safe_divide

df_scores = pd.DataFrame(validation_scores)

columns=[
    "open",
    "high",
    "low",
    "close",
    "adjusted_close",
    "volume",
]

prices = load_bronze_daily_prices(
    engine=ENGINE,
    provider=PROVIDER,
    start_date=start_date,
    end_date=end_date,
    columns=columns,
    tickers=tickers,
).set_index(["provider", "ticker", "trading_date"])

prices

In [ ]:
def add_annotation(x, y, kind: str, value: float, fig: go.Figure) -> None:
    font_color = "#39da89" if kind == "low" else "#eb4343"
    fig.add_annotation(
        x=x,
        y=y,
        xref="x",
        yref="y",
        text=str(value),
        showarrow=True,
        font=dict(
            family="Courier New, monospace",
            size=10,
            color=font_color,
            weight=900,
        ),
        align="center",
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor="#636363",
        ax=0,
        ay=5 * np.log2(y) if kind == "low" else -5 * np.log2(y),
        bordercolor="#c7c7c7",
        borderwidth=1,
        borderpad=3,
        bgcolor="#484848",
        opacity=0.8,
        row=1, col=1,
    )

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
shuffled_tickers = rng.permutation(prices.index.get_level_values("ticker").unique())

grouped = prices.sort_index(level="trading_date").groupby("ticker", sort=False)
iterator = iter((t, grouped.get_group(t)) for t in shuffled_tickers)

In [ ]:
ticker, frame = next(iterator)
pivots = indicators.pivot_points_high_low(
    data=frame,
    high_left=15,
    high_right=15,
    low_left=15,
    low_right=15,
    rank_output="strength",
    kind="balanced",
)
frame = pd.concat([frame, pivots], axis=1)


fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.01, row_heights=[0.6, 0.4],
)

# -----------------------
# SUBPLOT #1
# -----------------------

fig.add_candlestick(
    x=frame.index.get_level_values("trading_date"),
    open=frame["open"],
    high=frame["high"],
    low=frame["low"],
    close=frame["close"],
    line_width=1,
    row=1,
    col=1,
)

atr_14 = indicators.atr(frame, length=14)
stop_price = (frame["open"] - atr_14.mul(1)).rename("stop_price")
tp_price = (frame["open"] + 2 * atr_14.mul(1)).rename("tp_price")

extra_lines = pd.concat(
    [
        indicators.ema(frame["close"], length=10).rename("ema10"),
        indicators.ema(frame["close"], length=20).rename("ema20"),
        indicators.ema(frame["close"], length=50).rename("ema50"),
        indicators.ema(frame["close"], length=150).rename("ema150"),
        indicators.ema(frame["close"], length=200).rename("ema200"),
        stop_price,
        tp_price,
    ],
    axis=1,
)
for name, series in extra_lines.items():
    fig.add_scatter(
        x=series.index.get_level_values("trading_date"),
        y=series.values,
        line_shape="hvh",
        line_width=1.5,
        name=name,
        row=1,
        col=1,
    )

# Plot high/low annotations
highs = frame.query("pivot_high")["high"]
lows = frame.query("pivot_low")["low"]

for ind, y in lows.items():
    x = ind[-1]
    y = round(y, 1)
    add_annotation(x, y, kind="low", value=y, fig=fig)
for ind, y in highs.items():
    x = ind[-1]
    y = round(y, 1)
    add_annotation(x, y, kind="high", value=y, fig=fig)


# -----------------------
# SUBPLOT #2
# -----------------------
current_scores = df_scores.query("ticker == @ticker")["score"]
relative_trailing_score = safe_divide(
    indicators.ema(current_scores, length=2),
    indicators.sma(current_scores, length=6),
).apply(np.log).rename("relative_trailing_score")

extra_lines = pd.concat(
    [
        current_scores,
        indicators.ema(current_scores, length=5).rename("ema(scores, 5)"),
        relative_trailing_score,
        # features.donchian_position(frame, length=50),
    ],
    axis=1,
).astype(float)

for name, series in extra_lines.items():
    fig.add_scatter(
        x=series.index.get_level_values("trading_date"),
        y=series.values,
        line_shape="hv",
        name=name,
        mode="lines+markers",
        marker_size=1,
        row=2,
        col=1,
    )

fig.update_layout(
    xaxis_rangeslider_visible=False,
    title_text=ticker,
    height=750,
    hovermode="x unified",
    hoversubplots="axis",
)
fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.hist(
    df_scores["score"],
    bins="auto",
    histtype="step",
    linewidth=2,
    cumulative=False,
    density=False,
    log=False,
    zorder=3,
)
ax.axvline(0, color="k", linewidth=1, zorder=2)
ax.grid(zorder=1)
ax.set_xticks(np.arange(-1, 1.51, 0.1))
ax.tick_params(axis="x", rotation=45)
plt.show()

## Save a study result

For a material tuning run, export the executed notebook to HTML under `notebooks/exports/`. Record the iteration summary, chosen trial ID, feature columns, parameters, fold leaderboard, generalization gaps, and outer-validation tables in that export. Then restore this notebook to its generic base state.

The locked 2024–2025 test period remains untouched until the feature schema, hyperparameters, candidate rule, and model family are frozen.
